In [1]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen2.5:latest",
    temperature=0.5,
    max_tokens=250
)

tc_llm = ChatOllama(
    base_url="http://localhost:11434",
    model="llama3.2:latest",
    temperature=0.5,
    max_tokens=250
)

In [2]:
from langchain.tools import tool
from langchain_core.prompts import PromptTemplate


@tool
def generate_test_cases(user_story: str) -> str:
    "Generates test case from the user stories"
    prompt_template = PromptTemplate.from_template(
        
        """
        You are are QA Automation Engineer.
        Your task is to convert the following user stories into at least 10 testcases, in Gherkin BDD style format.
        Include combinations of valid, invalid, edge case and alternative flow scearios
        
        User Stories: {user_story}
        
        Format:
        Feature: ...
        Scenario: ...
            Given ...
            When ...
            Then ...
        
        """
    )
    prompt = prompt_template.invoke({"user_story":user_story })
    return tc_llm.invoke(prompt)
    

In [ ]:
from langchain_classic.agents import initialize_agent, AgentType

agent = initialize_agent(
    tools= [generate_test_cases],
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)

user_story_input = """
    As a user, I want to reset my password using a link sent to my email, so that I can
    regain access to my account if I forgot my password.
"""

result = agent.invoke(f"Create BDD Test cases for: {user_story_input}")

print(result)


/var/folders/jc/c7p4f2sd36xbqwkp2djn8flc0000gn/T/ipykernel_1400/2633484747.py:3: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the [LangGraph documentation](https://langchain-ai.github.io/langgraph/) as well as guides for [Migrating from AgentExecutor](https://python.langchain.com/docs/how_to/migrate_agent/) and LangGraph's [Pre-built ReAct agent](https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/).
  agent = initialize_agent(




> Entering new AgentExecutor chain...
Action:
```
{
  "action": "generate_test_cases",
  "action_input": {
    "user_story": "As a user, I want to reset my password using a link sent to my email, so that I can regain access to my account if I forgot my password."
  }
}
```
Observation: content='Here are 10 testcases in Gherkin BDD style format for the given user story:\n\n**Feature: Reset Password via Email**\n\n**Scenario 1: Valid User with Existing Account**\n```gherkin\nFeature: Reset Password via Email\n  Scenario: Valid user with existing account\n    Given I am a registered user with email "user@example.com" and password "password123"\n    When I request password reset link for my email address\n    Then I should receive an email with password reset link\n\nScenario 2: Valid User without Existing Account**\n```gherkin\nFeature: Reset Password via Email\n  Scenario: Valid user without existing account\n    Given I am a registered user with email "user@example.com" and password "